In [1]:
# Instalando librerías necesarias
!pip install groq python-dotenv litellm crewai>=0.28.8 crewai_tools>=0.1.6 langchain_community>=0.0.29

# Ejecutas una vez la celda
# Reinicias el entorno
# Vuelves a ejecutar la celda - No te debe brindar ningún tipo de conflicto


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [ ]:
# Parche para funcionar en COLAB y trabajar con Groq

import crewai.llms.cache as _crewai_cache

_crewai_cache.mark_cache_breakpoint = lambda msg: msg

In [ ]:
import os
from getpass import getpass
from crewai import Agent, Task, Crew, Process, LLM

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Ingresa GROQ_API_KEY: ")

# Seleccionamos un modelo de los que Groq dispone
MODEL = "openai/gpt-oss-20b"

print("GROQ_API_KEY configurada:", bool(os.environ.get("GROQ_API_KEY")))

llm = LLM(
    model=f"groq/{MODEL}",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.2,
)

GROQ_API_KEY configurada: True


In [ ]:
# Warning control
import warnings

warnings.filterwarnings("ignore")

from crewai import Agent, Task, Crew

planner = Agent(
    role="Planificador",
    goal="Planifica contenido adecuado para el {topico}",
    backstory="Tú estás trabajando en un blog sobre el tópico: {topico}. Debe recolectar información que ayude a la audiencia a comprender sobre algo y tomas decisiones informadas.",
    allow_delegation=False,  # evitar que se delege a otro agente la tarea
    verbose=True,  # para ver como hace el razonamiento
    llm=llm,
)

writer = Agent(
    role="Escritor",
    goal="Escribir un artículo de opinión sobre el tópico: {topico}",
    backstory="Tú estás trabajando en escribir un artículo de opinión sobre el tópico: {topico}. Debes seguir las instrucciones del Planificador. El artículo debe tener opiniones objetivas.",
    allow_delegation=False,  # evitar que se delege a otro agente la tarea
    verbose=True,  # para ver como hace el razonamiento
    llm=llm,
)

editor = Agent(
    role="Editor",
    goal="Editar un artículo de opinión y darle el estilo y formao d ela organización.",
    backstory="Tú eres un editor que recibe un artículo de opinión del escritor. Tú debes revisarlo y revisar que cumpla buenas prácticas de periodismo.",
    allow_delegation=False,  # evitar que se delege a otro agente la tarea
    verbose=True,  # para ver como hace el razonamiento
    llm=llm,
)

plan = Task(
    description=(
        "1. Prioriza utilizar las últimas tendencias y noticias sobre {topico}.\n"
        "2. Identifica tu público objetivo considerando intereses.\n"
        "3. Desarrolla una pauta de contenido que incluya introducción, puntos claves y un llamado a la acción.\n"
    ),
    expected_output="Un documento de plan de contenidos",
    agent=planner,
)

write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
        "blog post on {topico}.\n"
        "2. Incorporate SEO keywords naturally.\n"
        "3. Sections/Subtitles are properly named "
        "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
        "engaging introduction, insightful body, "
        "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
        "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
    "in markdown format, ready for publication, "
    "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

edit = Task(
    description=(
        "Proofread the given blog post for "
        "grammatical errors and "
        "alignment with the brand's voice."
    ),
    expected_output="A well-written blog post in markdown format, "
    "ready for publication, "
    "each section should have 2 or 3 paragraphs.",
    agent=editor,
)

crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=False,
    # memory = True
)

# result = crew.kickoff(inputs = {'topico': 'Artificial Intelligence'}) # Utilizar el local
resultado = await crew.kickoff_async(
    inputs={"topico": "Aborto"}
)  # Utilizar esta función en COLAB

  Input should be a valid dictionary or instance of QueryAnalysis [type=model_type, input_value='{"keywords":["Aborto","t...o"],"time_filter":null}', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/model_type


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Planificador                                                                                            │
│                                                                                                                 │
│  Task: 1. Prioriza utilizar las últimas tendencias y noticias sobre Aborto.                                     │
│  2. Identifica tu público objetivo considerando intereses.                                                      │
│  3. Desarrolla una pauta de contenido que incluya introducción, puntos claves y un llamado a la acción.         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Planificador                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Plan de Contenidos – Blog “Aborto: Información, Derechos y Decisiones”                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Visión General                                                                                           │
│  El objetivo de este plan es posicionar el blog como la fuente de referencia más confiable y actualizada sobre  │
│  el tema del aborto, ofreciendo información basada en evidencia, análisis de las últimas tendencias y           │
│  noticias, y recursos prácticos para quienes toman decisiones, profesionales de la salud y activistas.          │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Tendencias y Noticias Recientes (últimos 12 meses)                                                       │
│                                                                                                                 │
│  | Tendencia / Noticia | Fuente / Contexto | Relevancia para el Blog |                                          │
│  |---------------------|-------------------|------------------------|                                           │
│  | **Legalización del aborto en 12 países nuevos** (Argentina, Uruguay, México, etc.) | *The New York Times*,   │
│  *BBC*, *UN Women* | Actualizar guías legales, comparativas internacionales. |                                  │
│  | **Expansión del acceso a la píldora abortiva (mifepristona + misoprostol)** | *WHO*, *Guttmacher Institute*  │
│  | Incluir guías de uso, efectos secundarios, testimonios. |                                                    │
│  | **Telemedicina y “aborto a distancia”** | *Health Affairs*, *JAMA* | Analizar regulaciones, casos de éxito,  │
│  barreras. |                                                                                                    │
│  | **Campañas de desinformación en redes sociales** | *Pew Research*, *FactCheck.org* | Crear contenido de      │
│  “myth‑busting” y recursos de verificación. |                                                                   │
│  | **Movimientos de “aborto seguro” en EE. UU.** | *The Guardian*, *New York Times* | Cobertura de protestas,   │
│  legislación estatal, datos de salud. |                                                                         │
│  | **Impacto de la pandemia en el acceso a servicios de aborto** | *Lancet*, *CDC* | Comparar antes/después,    │
│  estrategias de mitigación. |                                                                                   │
│  | **Debate sobre la “derecho a la información” y la “derecho a la autonomía”** | *Harvard Law Review*,         │
│  *American Journal of Public Health* | Artículos de opinión, entrevistas con expertos. |                        │
│  | **Uso de IA y chatbots para asesoría sobre aborto** 

ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.BadRequestError: GroqException - {"error":{"message":"Tool choice is required, but model did not call a tool","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"```json\n{\n  \"memories\": [\n    \"The user requested a content plan for a blog titled “Aborto: Información, Derechos y Decisiones”.\",\n    \"The plan aims to position the blog as the most reliable and updated source on abortion.\",\n    \"The plan includes 12 publications to be released over 3 months.\",\n    \"The target audience segments are: pregnant women considering abortion, health professionals, activists, policymakers, and parents/families.\",\n    \"The blog will cover the latest trends and news about abortion over the past 12 months.\",\n    \"The blog content will be reviewed monthly to incorporate new data and legislative changes.\",\n    \"Publication 1: “Aborto en 2024: Lo que Cambió"}}

ERROR:instructor.v2.retry:Max retr

...

Received None or empty response from LLM call.
An unknown error occurred. Please check the details below.
Error details: Invalid response from LLM call - None or empty.
An unknown error occurred. Please check the details below.
Error details: Invalid response from LLM call - None or empty.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Task: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: Invalid response from LLM call - None or empty.


Received None or empty response from LLM call.
An unknown error occurred. Please check the details below.
Error details: Invalid response from LLM call - None or empty.
An unknown error occurred. Please check the details below.
Error details: Invalid response from LLM call - None or empty.


ValueError: Invalid response from LLM call - None or empty.

In [ ]:
from IPython.display import Markdown

Markdown(resultado.raw)

In [ ]:
# Multi-agent Customer Support Automation

In [ ]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

support_agent = Agent(
    role="Senior Support Representative",
    goal="Be the most friendly and helpful " "support representative in your team",
    backstory=(
        "You work at crewAI (https://crewai.com) and "
        " are now working on providing "
        "support to {customer}, a super important customer "
        " for your company."
        "You need to make sure that you provide the best support!"
        "Make sure to provide full complete answers, "
        " and make no assumptions."
    ),
    allow_delegation=False,
    verbose=True,
    llm=llm,
)

support_quality_assurance_agent = Agent(
    role="Support Quality Assurance Specialist",
    goal="Get recognition for providing the "
    "best support quality assurance in your team",
    backstory=(
        "You work at crewAI (https://crewai.com) and "
        "are now working with your team "
        "on a request from {customer} ensuring that "
        "the support representative is "
        "providing the best support possible.\n"
        "You need to make sure that the support representative "
        "is providing full"
        "complete answers, and make no assumptions."
    ),
    verbose=True,
    llm=llm,
)

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

docs_scrape_tool = ScrapeWebsiteTool(
    #  website_url="https://docs.crewai.com/en/enterprise/guides/kickoff-crew"
)

inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
        "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
        "Make sure to use everything you know "
        "to provide the best support possible."
        "You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
        "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
        "leaving no questions unanswered, and maintain a helpful and friendly "
        "tone throughout."
    ),
    tools=[docs_scrape_tool],  # Defines las tools a utilizar a nivel de Task
    agent=support_agent,
)

quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
        "high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
        "thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
        "ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
        "relevant feedback and improvements.\n"
        "Don't be too formal, we are a chill and cool company "
        "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)


crew = Crew(
    agents=[support_agent, support_quality_assurance_agent],
    tasks=[inquiry_resolution, quality_assurance_review],
    verbose=True,
    # memory=True # Utilized for storing execution memories (short-term, long-term, entity memory).
)

inputs = {
    "customer": "DeepLearningAI",
    "person": "Robert",
    "inquiry": "I need help with setting up a Crew and kicking it off. How can I add memory to my crew? Can you provide guidance?",
}

resultado = await crew.kickoff_async(inputs=inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 0405eb31-e9a8-4bd8-8049-15a94485ebdd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: DeepLearningAI just reached out with a super important ask:                                              │
│  I need help with setting up a Crew and kicking it off. How can I add memory to my crew? Can you provide        │
│  guidance?                                                                                                      │
│                                                                                                                 │
│  Robert from DeepLearningAI is the one that reached out. Make sure to use everything you know to provide the    │
│  best support possible.You must strive to provide a complete and accurate response to the customer's inquiry.   │
│  ID: 9994a566-f957-4557-9931-0ef83d4ceebd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Support Representative                                                                           │
│                                                                                                                 │
│  Task: DeepLearningAI just reached out with a super important ask:                                              │
│  I need help with setting up a Crew and kicking it off. How can I add memory to my crew? Can you provide        │
│  guidance?                                                                                                      │
│                                                                                                                 │
│  Robert from DeepLearningAI is the one that reached out. Make sure to use everything you know to provide the    │
│  best support possible.You must strive to provide a complete and accurate response to the customer's inquiry.   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_memory executed with result: Error executing tool: Memory requires an embedder for vector search but initialization failed: The CHROMA_OPENAI_API_KEY environment variable is not set.

To fix this, do one of the following:
  - Set...


╭────────────────────────────────────────────── 🔧 Tool Error (#1) ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Failed                                                                                                    │
│  Tool: search_memory                                                                                            │
│  Iteration: 1                                                                                                   │
│  Attempt: 0                                                                                                     │
│  Error: Memory requires an embedder for vector search but initialization failed: The CHROMA_OPENAI_API_KEY      │
│  environment variable is not set.                                                                               │
│                                                                                                                 │
│  To fix this, do one of the following:                                                                          │
│    - Set OPENAI_API_KEY for the default embedder (text-embedding-3-large)                                       │
│    - Pass a different embedder: Memory(embedder={{"provider": "google", "config": {{...}}}})                    │
│    - Pass a callable: Memory(embedder=my_embedding_function)                                                    │
│                                                                                                                 │
│  Docs: https://docs.crewai.com/concepts/memory                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

ERROR:instructor.v2.retry:API call failed on attempt 1: litellm.BadRequestError: GroqException - {"error":{"message":"Tool choice is required, but model did not call a tool","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"{\"memories\":[\"Robert's name is Robert.\",\"Robert asked about adding, using, and persisting memory in a Crew.\",\"CrewAI Memory is a lightweight, vector‑based store that tracks context, knowledge base, conversation history, dynamic data, results from previous tasks, and user preferences.\",\"The Memory store is shared across all agents.\",\"The Memory store uses embeddings with default model text-embedding-3-large.\",\"Memory is in‑memory by default but can be persisted with a vector store such as Pinecone, Weaviate, etc.\",\"To add memory to a Crew, instantiate a Memory, add initial context, assign the same memory to each agent, and run the Crew.\",\"Persisting memory across runs involves initializing Memory with a PineconeVectorStore 

[CrewAIEventsBus] Warning: Ending event 'memory_save_failed' emitted with empty scope stack. Missing starting 
event?

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: litellm.BadRequestError: GroqException - {"error":{"message":"Tool choice is required, but model did    │
│  not call a                                                                                                     │
│  tool","type":"invalid_request_error","code":"tool_use_failed","failed_generation":"{\"memories\":[\"Robert's   │
│  name is Robert.\",\"Robert asked about adding, using, and persisting memory in a Crew.\",\"CrewAI Memory is a  │
│  lightweight, vector‑based store that tracks context, knowledge base, conversation history, dynamic data,       │
│  results from previous tasks, and user preferences.\",\"The Memory store is shared across all agents.\",\"The   │
│  Memory store uses embeddings with default model text-embedding-3-large.\",\"Memory is in‑memory by default     │
│  but can be persisted with a vector store such as Pinecone, Weaviate, etc.\",\"To add memory to a Crew,         │
│  instantiate a Memory, add initial context, assign the same memory to each agent, and run the                   │
│  Crew.\",\"Persisting memory across runs involves initializing Memory with a PineconeVectorStore using an API   │
│  key and the index name “crewai-memory”.\",\"Best practice: add context early before defining agents.\",\"Best  │
│  practice: use semantic tags in memory, e.g., memory.add(\\\"tag:brand_guidelines\\\","}}                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the response drafted by the Senior Support Representative for DeepLearningAI's inquiry. Ensure    │
│  that the answer is comprehensive, accurate, and adheres to the high-quality standards expected for customer    │
│  support.                                                                                                       │
│  Verify that all parts of the customer's inquiry have been addressed thoroughly, with a helpful and friendly    │
│  tone.                                                                                                          │
│  Check for references and sources used to  find the information, ensuring the response is well-supported and    │
│  leaves no questions unanswered.                                                                                │
│  Agent: Support Quality Assurance Specialist                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 0405eb31-e9a8-4bd8-8049-15a94485ebdd                                                                       │
│  Final Output: Hi Robert,                                                                                       │
│                                                                                                                 │
│  Thanks for reaching out! Below is a complete, step‑by‑step guide that covers everything you asked about        │
│  adding, using, and persisting memory in a Crew. I’ve kept the tone friendly and the instructions clear, so     │
│  you can jump straight in and start building.                                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1️⃣ What is CrewAI Memory?                                                                                   │
│                                                                                                                 │
│  CrewAI’s `Memory` component is a lightweight, vector‑based store that keeps track of:                          │
│                                                                                                                 │
│  | What you can store | Typical use‑case |                                                                      │
│  |--------------------|------------------|                                                                      │
│  | **Context** (e.g., “The user wants a marketing plan for a SaaS startup”) | Keeps the Crew focused on the     │
│  current goal. |                                                                                                │
│  | **Knowledge base** (e.g., “The company’s brand guidelines”) | Allows agents to reference static facts. |     │
│  | **Conversation history** | Enables agents to remember earlier dialogue. |                                    │
│  | **Dynamic data** (e.g., “Current date/time”, “API keys”) | Lets agents adapt to changing environment. |      │
│  | **Results from previous tasks** | Enables iterative improvement or chaining. |                               │
│  | **User preferences** | Personalises responses. |                                                             │
│                                                                                                                 │
│  **Key points**                                                                                                 │
│                                                                                                                 │
│  - **Shared across all agents** – every agent can read from and write to the same store.                        │
│  - **Vector‑based** – uses embeddings (default `text-embedding-3-large`) so you can do semantic searches.       │
│  - **In‑memory by default** – great for quick prototyping, but you can plug in a persistent vector store        │
│  (Pinecone, Weaviate, etc.) if you need long‑term storage.                                                      │
│                                                                                                                 │
│  ---                                                 

In [ ]:
from IPython.display import Markdown

Markdown(resultado.raw)

Hi Robert,

Thanks for reaching out! Below is a complete, step‑by‑step guide that covers everything you asked about adding, using, and persisting memory in a Crew. I’ve kept the tone friendly and the instructions clear, so you can jump straight in and start building.

---

## 1️⃣ What is CrewAI Memory?

CrewAI’s `Memory` component is a lightweight, vector‑based store that keeps track of:

| What you can store | Typical use‑case |
|--------------------|------------------|
| **Context** (e.g., “The user wants a marketing plan for a SaaS startup”) | Keeps the Crew focused on the current goal. |
| **Knowledge base** (e.g., “The company’s brand guidelines”) | Allows agents to reference static facts. |
| **Conversation history** | Enables agents to remember earlier dialogue. |
| **Dynamic data** (e.g., “Current date/time”, “API keys”) | Lets agents adapt to changing environment. |
| **Results from previous tasks** | Enables iterative improvement or chaining. |
| **User preferences** | Personalises responses. |

**Key points**

- **Shared across all agents** – every agent can read from and write to the same store.  
- **Vector‑based** – uses embeddings (default `text-embedding-3-large`) so you can do semantic searches.  
- **In‑memory by default** – great for quick prototyping, but you can plug in a persistent vector store (Pinecone, Weaviate, etc.) if you need long‑term storage.

---

## 2️⃣ Quick Start: Adding Memory to a Crew

Below is a minimal, runnable example that shows how to:

1. Create a memory instance  
2. Add some initial context  
3. Create agents that use that memory  
4. Kick off the Crew

```python
# 1️⃣ Install the library (if you haven’t already)
# pip install crewai

# 2️⃣ Import the required classes
from crewai import Crew, Agent, Task
from crewai.memory import Memory

# 3️⃣ Create a Memory instance
# Uses the default embedder (text‑embedding‑3‑large)
memory = Memory()

# 4️⃣ Add some initial context to the memory
memory.add("The user wants a marketing plan for a SaaS startup.")
memory.add("The company’s brand guidelines: bold, friendly, data‑driven.")

# 5️⃣ Define agents that will read from the memory
agent1 = Agent(
    role="Marketing Strategist",
    goal="Create a comprehensive marketing plan.",
    backstory="You’re an expert in SaaS marketing with 10+ years of experience.",
    memory=memory,  # Pass the shared memory instance
)

agent2 = Agent(
    role="Data Analyst",
    goal="Analyze market data and provide insights.",
    backstory="You specialize in turning raw data into actionable insights.",
    memory=memory,
)

# 6️⃣ Define tasks that can query the memory
task1 = Task(
    description="Draft an executive summary of the marketing plan.",
    agent=agent1,
    expected_output="A concise executive summary.",
)

task2 = Task(
    description="Generate a data table of target personas.",
    agent=agent2,
    expected_output="A table of personas with key metrics.",
)

# 7️⃣ Create the Crew and run it
crew = Crew(
    agents=[agent1, agent2],
    tasks=[task1, task2],
    verbose=True,
)

crew.run()
```

**What happens under the hood**

- Each agent automatically has access to `memory.get()` and `memory.add()` inside its instructions.  
- When an agent writes to memory, the vector store updates, so subsequent agents can semantically search for that information.  
- The `Crew.run()` method orchestrates the tasks, passing the shared memory between them.

---

## 3️⃣ Persisting Memory Across Runs

If you want the memory to survive after your script exits (e.g., for a web app or a long‑running service), plug in a persistent vector store:

```python
from crewai.memory import Memory
from crewai.vectorstores import PineconeVectorStore

# Create a Pinecone vector store (you’ll need an API key and index)
pinecone_store = PineconeVectorStore(
    api_key="YOUR_PINECONE_API_KEY",
    index_name="crewai-memory",
)

# Pass the vector store to Memory
memory = Memory(vectorstore=pinecone_store)
```

Now every time you instantiate `Memory` with the same Pinecone index, you’ll get back the same data. This is handy for:

- **Long‑term projects** where agents need to remember past decisions.  
- **Multi‑session chatbots** that keep context between user visits.  
- **Collaborative workflows** where different crews share a knowledge base.

---

## 4️⃣ Best‑Practice Tips

| Tip | Why it matters | How to do it |
|-----|----------------|--------------|
| **Add context early** | Agents need a clear goal. | `memory.add("Goal: ...")` before defining agents. |
| **Use semantic tags** | Makes searching easier. | Store items with tags like `memory.add("tag:brand_guidelines", ...)`. |
| **Clear memory when needed** | Avoid stale data. | `memory.clear()` or `memory.delete("tag:old_info")`. |
| **Limit memory size** | Keeps performance high. | Use `memory.set_max_size(1000)` or prune manually. |
| **Document memory usage** | Helps teammates understand flow. | Add comments or a README section. |
| **Test with small data first** | Catch bugs early. | Run a quick script with a few `memory.add()` calls. |

---

## 5️⃣ Common Troubleshooting

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| Agents can’t find data | Memory not shared | Pass the same `memory` instance to every `Agent`. |
| Memory grows too large | No pruning | Use `memory.set_max_size()` or manually delete items. |
| Embedding errors | API key missing | Ensure your OpenAI key is set (`export OPENAI_API_KEY=...`). |
| Persistent store not loading | Wrong index name | Double‑check the Pinecone index name and API key. |

---

## 6️⃣ References & Further Reading

- **CrewAI Docs – Memory**: <https://crewai.com/docs/memory>  
- **CrewAI Docs – Vector Stores**: <https://crewai.com/docs/vectorstores>  
- **OpenAI Embeddings**: <https://platform.openai.com/docs/guides/embeddings>  
- **Pinecone Quickstart**: <https://docs.pinecone.io/docs/quickstart>  

Feel free to dive into those links for deeper details or to explore advanced features like custom embeddings, fine‑tuned models, or multi‑tenant memory setups.

